# AIA 2026 Missing-Only Recovery Miner

This notebook performs a **surgical recovery** of the 262 unresolved 2026 samples without rerunning the full 3,201-sample production plan.

## Confirmed recovery classes

- **217 samples** in 16 blocks with wavelength download/time-coverage failures.
- **45 samples** from HARP 14305 with crop-boundary failures caused by an undersized tracked patch.

## Recovery strategy

1. Read `recovery_inventory_aia2026.csv`.
2. Use the existing GCP objects as the source of truth and skip anything already present.
3. Reuse retained production FITS files where they already cover the target times.
4. For coverage failures, request a denser **12-minute sequence** only for wavelengths that remain uncovered.
5. If a target is still uncovered, use a narrow **1-minute-cadence micro-window** around that target.
6. For crop-boundary failures, request a substantially larger tracked patch.
7. Keep the scientific AIA–SHARP time tolerance fixed at **180 seconds**.
8. Upload each successful `(512, 512, 6)` tensor immediately and checkpoint every result.

Run `RECOVERY_MODE=CANARY` first. It selects one group from each failure class. After checking the outputs, rerun with `RECOVERY_MODE=PRODUCTION`.

In [ ]:
import os
import re
import gc
import sys
import json
import time
import math
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

import drms
from drms.exceptions import DrmsExportError
from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord
from skimage.transform import resize
from IPython.display import display

import sunpy
import sunpy.map

print("Python:", sys.version)
print("DRMS:", drms.__version__)
print("SunPy:", sunpy.__version__)

## 1. Configuration

In [ ]:
PROJECT_ID = "sonorous-shore-450510-i4"
GCP_BUCKET = "gs://suryabench-sharp-pipeline-bamidele"

TARGET_YEAR = 2026
AIA_WAVELENGTHS = [94, 131, 171, 193, 211, 335]
IMAGE_SIZE = 512

JSOC_EMAIL = os.environ.get("JSOC_EMAIL", "worky4work@gmail.com")
WORKER_ID = os.environ.get("WORKER_ID", "aia2026-recovery")
RECOVERY_MODE = os.environ.get("RECOVERY_MODE", "CANARY").upper()

if RECOVERY_MODE not in {"CANARY", "PRODUCTION"}:
    raise ValueError("RECOVERY_MODE must be CANARY or PRODUCTION.")

MAX_GROUPS_THIS_RUN = (
    int(os.environ["MAX_GROUPS_THIS_RUN"])
    if os.environ.get("MAX_GROUPS_THIS_RUN")
    else None
)

# Scientific time-matching rule: do not relax this.
MAX_TARGET_TIME_DIFFERENCE_SEC = 180

# Recovery sequence cadence. The normal miner used 96 minutes.
# A denser 12-minute export is used only for unresolved blocks.
RECOVERY_SEQUENCE_CADENCE_MIN = 12

# If the dense block sequence still misses a timestamp, request a small
# target-centred window sampled every minute.
MICRO_WINDOW_BEFORE_MIN = 4
MICRO_WINDOW_DURATION_MIN = 8
MICRO_WINDOW_CADENCE_MIN = 1

# Larger server-side patches for recovery.
NORMAL_RECOVERY_MARGIN_ARCSEC = 240.0
CROP_RECOVERY_MARGIN_ARCSEC = 480.0
MIN_RECOVERY_PATCH_ARCSEC = 360.0
MAX_RECOVERY_PATCH_ARCSEC = 2400.0

# Historical geometry retained from the production miner.
FULL_DISK_SIZE = 4096
IMAGE_CENTER = FULL_DISK_SIZE // 2
AIA_PIXEL_SCALE_ARCSEC = 0.6
SOLAR_RADIUS_ARCSEC = 976.0
SOLAR_RADIUS_PIX = SOLAR_RADIUS_ARCSEC / AIA_PIXEL_SCALE_ARCSEC
CROP_SCALE = 1.2
CROP_PADDING_PIX = 30
MIN_CROP_PIX = 64

# JSOC queue protection.
JSOC_MAX_RETRIES = 10
JSOC_INITIAL_BACKOFF_SEC = 20
JSOC_MAX_BACKOFF_SEC = 300
JSOC_WAIT_TIMEOUT_SEC = 7200
JSOC_COOLDOWN_SEC = 12

MIN_FREE_DISK_GB = 15

BASE = Path.home() / "solar_flare_aia"
PRODUCTION_ROOT = BASE / "harp_block_miner" / "production_aia2026"
SOURCE_TEMP_ROOT = PRODUCTION_ROOT / "temp_blocks"
SOURCE_META_ROOT = PRODUCTION_ROOT / "metadata"

LOCAL_ROOT = BASE / "harp_block_miner" / f"recovery_{WORKER_ID}"
LOCAL_META = LOCAL_ROOT / "metadata"
LOCAL_TEMP = LOCAL_ROOT / "temp_blocks"
LOCAL_OUTPUT = LOCAL_ROOT / "samples_npz" / str(TARGET_YEAR)

for directory in [LOCAL_ROOT, LOCAL_META, LOCAL_TEMP, LOCAL_OUTPUT]:
    directory.mkdir(parents=True, exist_ok=True)

LOCAL_RECOVERY_LOG = LOCAL_META / f"recovery_sample_log_{WORKER_ID}.csv"
LOCAL_GROUP_LOG = LOCAL_META / f"recovery_group_log_{WORKER_ID}.csv"
LOCAL_PLAN = LOCAL_META / f"recovery_plan_{WORKER_ID}.csv"

GCP_RUN_ROOT = f"{GCP_BUCKET}/jsoc_2025_2026_production_v1"
GCP_OUTPUT_ROOT = f"{GCP_RUN_ROOT}/samples_npz/{TARGET_YEAR}"
GCP_WORKER_META = f"{GCP_RUN_ROOT}/metadata/workers/{WORKER_ID}"

GCP_METADATA_CANDIDATES = [
    f"{GCP_BUCKET}/metadata/curated_sharp_suryabench_true96min_48h_AR_SPECIFIC_2010_2026.csv",
    f"{GCP_BUCKET}/metadata/curated_2025_2026_AR_SPECIFIC_EXTENSION.csv",
]

INVENTORY_LOCAL = SOURCE_META_ROOT / "recovery_inventory_aia2026.csv"
INVENTORY_GCP = (
    f"{GCP_RUN_ROOT}/metadata/workers/aia2026/"
    "recovery_inventory_aia2026.csv"
)

print("=" * 88)
print("RECOVERY_MODE:", RECOVERY_MODE)
print("JSOC_EMAIL:", JSOC_EMAIL)
print("WORKER_ID:", WORKER_ID)
print("LOCAL_ROOT:", LOCAL_ROOT)
print("SOURCE_TEMP_ROOT:", SOURCE_TEMP_ROOT)
print("GCP_OUTPUT_ROOT:", GCP_OUTPUT_ROOT)
print("=" * 88)

## 2. Cloud, JSOC and file utilities

In [ ]:
def run_command(command, check=True, capture=True):
    result = subprocess.run(
        command,
        text=True,
        capture_output=capture,
    )
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(command)}\n"
            f"{result.stderr[-3000:] if result.stderr else ''}"
        )
    return result


def gcp_exists(path):
    return run_command(
        ["gcloud", "storage", "ls", path],
        check=False,
    ).returncode == 0


def copy_first_existing(candidates, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    for candidate in candidates:
        print("Checking:", candidate)
        if not gcp_exists(candidate):
            continue
        run_command(
            ["gcloud", "storage", "cp", candidate, str(destination)],
            check=True,
        )
        if destination.exists() and destination.stat().st_size > 0:
            print("✅ Copied:", candidate)
            return candidate
    raise FileNotFoundError(
        "None of the required GCP files could be found."
    )


def upload_verified(local_path, gcp_path):
    run_command(
        ["gcloud", "storage", "cp", str(local_path), gcp_path],
        check=True,
    )
    if not gcp_exists(gcp_path):
        raise RuntimeError(f"Upload verification failed: {gcp_path}")


def free_disk_gb(path):
    usage = shutil.disk_usage(path)
    return usage.free / (1024 ** 3)


print("Testing bucket access...")
run_command(["gcloud", "storage", "ls", GCP_BUCKET], check=True)
print("✅ Bucket access works.")

registered = drms.Client().check_email(JSOC_EMAIL)
print("JSOC registered:", registered, "|", JSOC_EMAIL)
if not registered:
    raise RuntimeError(f"JSOC email is not registered: {JSOC_EMAIL}")

jsoc = drms.Client(email=JSOC_EMAIL)
print("✅ JSOC client initialised.")

## 3. Load the corrected metadata and exact recovery inventory

In [ ]:
def clean_noaa(value):
    if pd.isna(value):
        return np.nan
    try:
        number = int(float(value))
        return number if number > 0 else np.nan
    except Exception:
        matches = re.findall(r"\d+", str(value))
        return int(matches[0]) if matches else np.nan


metadata_path = LOCAL_META / "corrected_ar_specific_metadata.csv"
copy_first_existing(GCP_METADATA_CANDIDATES, metadata_path)

if not INVENTORY_LOCAL.exists():
    copy_first_existing([INVENTORY_GCP], INVENTORY_LOCAL)

raw_df = pd.read_csv(metadata_path, low_memory=False)
inventory = pd.read_csv(INVENTORY_LOCAL, low_memory=False)

raw_df["T_REC_dt"] = pd.to_datetime(raw_df["T_REC_dt"], errors="coerce")

if "NOAA_AR_clean" not in raw_df.columns:
    source_column = (
        "NOAA_ARS"
        if "NOAA_ARS" in raw_df.columns
        else "NOAA_AR"
    )
    raw_df["NOAA_AR_clean"] = raw_df[source_column].apply(clean_noaa)

label_source = next(
    (
        column
        for column in [
            "label_48h_final",
            "label_48h_ar_specific",
            "label_48h",
        ]
        if column in raw_df.columns
    ),
    None,
)
if label_source is None:
    raise ValueError("No AR-specific 48-hour label column exists.")

raw_df["label_48h_final"] = pd.to_numeric(
    raw_df[label_source],
    errors="coerce",
)
raw_df["HARPNUM"] = pd.to_numeric(raw_df["HARPNUM"], errors="coerce")
raw_df["NOAA_AR_clean"] = pd.to_numeric(
    raw_df["NOAA_AR_clean"],
    errors="coerce",
)

required = [
    "T_REC_dt",
    "HARPNUM",
    "NOAA_AR_clean",
    "label_48h_final",
    "LON_MIN",
    "LON_MAX",
    "LAT_MIN",
    "LAT_MAX",
]
missing_columns = [
    column for column in required if column not in raw_df.columns
]
if missing_columns:
    raise ValueError(f"Missing metadata columns: {missing_columns}")

raw_df = raw_df.dropna(subset=required).copy()
raw_df["HARPNUM"] = raw_df["HARPNUM"].astype(int)
raw_df["NOAA_AR_clean"] = raw_df["NOAA_AR_clean"].astype(int)
raw_df["label_48h_final"] = raw_df["label_48h_final"].astype(int)

if "sample_id" not in raw_df.columns:
    raw_df["sample_id"] = raw_df.apply(
        lambda row: (
            f"{row['T_REC_dt'].strftime('%Y%m%d_%H%M')}"
            f"_HARP{row['HARPNUM']}"
            f"_NOAA{row['NOAA_AR_clean']}"
        ),
        axis=1,
    )

metadata_2026 = (
    raw_df[raw_df["T_REC_dt"].dt.year == TARGET_YEAR]
    .drop_duplicates("sample_id")
    .copy()
)

if len(metadata_2026) != 3201:
    raise RuntimeError(
        f"Expected 3,201 curated 2026 rows; found {len(metadata_2026)}."
    )

inventory["sample_id"] = inventory["sample_id"].astype(str)
recovery_df = inventory[
    ["sample_id", "recovery_class", "block_id", "error_message"]
].merge(
    metadata_2026,
    on="sample_id",
    how="left",
    validate="one_to_one",
)

if recovery_df[required].isna().any().any():
    bad = recovery_df[
        recovery_df[required].isna().any(axis=1)
    ]["sample_id"].tolist()
    raise RuntimeError(
        f"Recovery inventory did not merge cleanly: {bad[:10]}"
    )

if len(recovery_df) != 262:
    raise RuntimeError(
        f"Expected 262 recovery rows; found {len(recovery_df)}."
    )

print("Recovery rows:", len(recovery_df))
print(recovery_df["recovery_class"].value_counts().to_string())
print("Unique original blocks:", recovery_df["block_id"].nunique())

group_table = (
    recovery_df.groupby(
        ["recovery_class", "block_id"],
        dropna=False,
    )
    .agg(
        HARPNUM=("HARPNUM", "first"),
        start=("T_REC_dt", "min"),
        end=("T_REC_dt", "max"),
        n_targets=("sample_id", "size"),
    )
    .reset_index()
    .sort_values(["recovery_class", "start", "HARPNUM"])
    .reset_index(drop=True)
)

group_table["recovery_group_id"] = group_table.apply(
    lambda row: (
        f"{row['recovery_class']}__{row['block_id']}"
    ),
    axis=1,
)

if RECOVERY_MODE == "CANARY":
    selected_rows = []
    for recovery_class in [
        "block_download_or_coverage",
        "crop_patch_boundary",
    ]:
        subset = group_table[
            group_table["recovery_class"] == recovery_class
        ]
        if len(subset):
            selected_rows.append(subset.head(1))
    group_table = pd.concat(selected_rows, ignore_index=True)

if MAX_GROUPS_THIS_RUN is not None:
    group_table = group_table.head(MAX_GROUPS_THIS_RUN).copy()

group_table.to_csv(LOCAL_PLAN, index=False)
upload_verified(
    LOCAL_PLAN,
    f"{GCP_WORKER_META}/{LOCAL_PLAN.name}",
)

selected_group_ids = set(group_table["recovery_group_id"])
recovery_df["recovery_group_id"] = recovery_df.apply(
    lambda row: f"{row['recovery_class']}__{row['block_id']}",
    axis=1,
)
selected_recovery_df = recovery_df[
    recovery_df["recovery_group_id"].isin(selected_group_ids)
].copy()

print("\nSelected recovery groups:", len(group_table))
print("Selected target rows:", len(selected_recovery_df))
display(group_table)

## 4. Geometry and preprocessing

In [ ]:
def lonlat_to_pixel(lon_deg, lat_deg):
    lon = np.deg2rad(float(lon_deg))
    lat = np.deg2rad(float(lat_deg))
    x = IMAGE_CENTER + SOLAR_RADIUS_PIX * np.cos(lat) * np.sin(lon)
    y = IMAGE_CENTER - SOLAR_RADIUS_PIX * np.sin(lat)
    return float(x), float(y)


def target_geometry(row):
    corners = [
        (row["LON_MIN"], row["LAT_MIN"]),
        (row["LON_MIN"], row["LAT_MAX"]),
        (row["LON_MAX"], row["LAT_MIN"]),
        (row["LON_MAX"], row["LAT_MAX"]),
    ]
    pixels = [lonlat_to_pixel(lon, lat) for lon, lat in corners]
    xs = [item[0] for item in pixels]
    ys = [item[1] for item in pixels]

    center_x_pix = (min(xs) + max(xs)) / 2.0
    center_y_pix = (min(ys) + max(ys)) / 2.0

    width_pix = max(max(xs) - min(xs), MIN_CROP_PIX)
    height_pix = max(max(ys) - min(ys), MIN_CROP_PIX)
    crop_pix = max(width_pix, height_pix) * CROP_SCALE + CROP_PADDING_PIX

    return {
        "x_arcsec": (center_x_pix - IMAGE_CENTER) * AIA_PIXEL_SCALE_ARCSEC,
        "y_arcsec": (IMAGE_CENTER - center_y_pix) * AIA_PIXEL_SCALE_ARCSEC,
        "box_arcsec": crop_pix * AIA_PIXEL_SCALE_ARCSEC,
    }


def historical_preprocess(image):
    image = np.asarray(image, dtype=np.float32)
    image = np.nan_to_num(
        image,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )
    image = np.clip(image, 0, None)
    image = np.log1p(image)

    low = float(image.min())
    high = float(image.max())
    if high <= low:
        return np.zeros_like(image, dtype=np.float32)

    return ((image - low) / (high - low)).astype(np.float32)


def crop_target_from_patch(fits_path, row):
    solar_map = sunpy.map.Map(fits_path)
    data = np.asarray(solar_map.data, dtype=np.float32)
    geometry = target_geometry(row)

    coordinate = SkyCoord(
        geometry["x_arcsec"] * u.arcsec,
        geometry["y_arcsec"] * u.arcsec,
        frame=solar_map.coordinate_frame,
    )
    pixel = solar_map.world_to_pixel(coordinate)
    center_x = float(pixel.x.value)
    center_y = float(pixel.y.value)

    scale_x = abs(
        float(solar_map.scale.axis1.to_value(u.arcsec / u.pix))
    )
    scale_y = abs(
        float(solar_map.scale.axis2.to_value(u.arcsec / u.pix))
    )
    half_width = geometry["box_arcsec"] / (2.0 * scale_x)
    half_height = geometry["box_arcsec"] / (2.0 * scale_y)

    x0 = int(math.floor(center_x - half_width))
    x1 = int(math.ceil(center_x + half_width))
    y0 = int(math.floor(center_y - half_height))
    y1 = int(math.ceil(center_y + half_height))

    if x0 < 0 or y0 < 0 or x1 > data.shape[1] or y1 > data.shape[0]:
        raise ValueError(
            "Target crop leaves recovery patch: "
            f"bounds={(x0, x1, y0, y1)}, shape={data.shape}"
        )

    crop = data[y0:y1, x0:x1]
    if crop.size == 0:
        raise ValueError("Empty recovery crop.")

    resized = resize(
        crop,
        (IMAGE_SIZE, IMAGE_SIZE),
        anti_aliasing=True,
        preserve_range=True,
    )

    return historical_preprocess(resized), {
        "patch_shape": list(data.shape),
        "local_bounds": [x0, x1, y0, y1],
        "target_geometry": geometry,
    }


print("✅ Geometry and preprocessing functions ready.")

## 5. Retry-safe JSOC exports and time matching

In [ ]:
def parse_jsoc_time(value):
    try:
        return pd.Timestamp(drms.to_datetime(str(value)))
    except Exception:
        text = str(value).replace("_TAI", "").replace("Z", "")
        return pd.to_datetime(text, errors="coerce")


def extract_request_id(message):
    match = re.search(r"(JSOC_\d{8}_\d+)", str(message))
    return match.group(1) if match else None


def wait_for_existing_request(request_id):
    print("Waiting for existing RequestID:", request_id)
    old_request = jsoc.export_from_id(request_id)
    old_request.wait(
        timeout=JSOC_WAIT_TIMEOUT_SEC,
        sleep=15,
        retries_notfound=30,
    )


def submit_export_retry_safe(query_string, process):
    delay = JSOC_INITIAL_BACKOFF_SEC
    last_error = None

    for attempt in range(1, JSOC_MAX_RETRIES + 1):
        try:
            print(
                f"JSOC export attempt {attempt}/{JSOC_MAX_RETRIES}"
            )
            request = jsoc.export(
                query_string,
                method="url",
                protocol="fits",
                email=JSOC_EMAIL,
                process=process,
            )
            request.wait(
                timeout=JSOC_WAIT_TIMEOUT_SEC,
                sleep=15,
                retries_notfound=30,
            )

            if not request.has_succeeded():
                raise RuntimeError(
                    f"Request failed: id={request.id}, "
                    f"status={request.status}"
                )
            return request

        except DrmsExportError as error:
            last_error = error
            message = str(error)

            if "pending export requests" not in message.lower():
                raise

            request_id = extract_request_id(message)
            print("JSOC pending-request protection triggered.")
            print(message)

            if request_id:
                try:
                    wait_for_existing_request(request_id)
                except Exception as wait_error:
                    print(
                        "Could not reopen old request:",
                        repr(wait_error),
                    )

            print(f"Sleeping {delay} seconds...")
            time.sleep(delay)
            delay = min(delay * 2, JSOC_MAX_BACKOFF_SEC)

    raise RuntimeError(
        f"JSOC remained busy after all retries: {last_error}"
    )


def format_query_time(timestamp):
    return pd.Timestamp(timestamp).strftime(
        "%Y-%m-%dT%H:%M:%S.000"
    )


def fits_observation_time(path):
    with fits.open(path, memmap=False) as hdul:
        headers = [
            hdu.header
            for hdu in hdul
            if getattr(hdu, "header", None) is not None
        ]

    for header in headers:
        for key in ["T_REC", "DATE-OBS", "DATE_OBS", "T_OBS"]:
            if key in header:
                parsed = parse_jsoc_time(header[key])
                if not pd.isna(parsed):
                    return pd.Timestamp(parsed)

    raise ValueError(f"No observation time found in {path}")


def index_downloaded_files(files):
    indexed = []
    for path in files:
        path = Path(path)
        try:
            timestamp = fits_observation_time(path)
            indexed.append((timestamp, path))
        except Exception as error:
            print(
                "Skipping unreadable FITS time:",
                path,
                repr(error),
            )

    if not indexed:
        raise RuntimeError(
            "No downloaded FITS file has a valid timestamp."
        )

    return sorted(indexed, key=lambda item: item[0])


def target_deltas(indexed_files, target_frame):
    available_times = [item[0] for item in indexed_files]
    results = {}

    for sample_id, target in zip(
        target_frame["sample_id"],
        pd.to_datetime(target_frame["T_REC_dt"]),
    ):
        nearest_delta = min(
            abs((timestamp - pd.Timestamp(target)).total_seconds())
            for timestamp in available_times
        )
        results[str(sample_id)] = float(nearest_delta)

    return results


def files_cover_targets(files, target_frame):
    files = [
        Path(item)
        for item in files
        if str(item).lower().endswith(".fits")
    ]
    if not files:
        return False

    try:
        indexed = index_downloaded_files(files)
    except Exception:
        return False

    return all(
        delta <= MAX_TARGET_TIME_DIFFERENCE_SEC
        for delta in target_deltas(indexed, target_frame).values()
    )


def nearest_file(indexed_files, target_time):
    target_time = pd.Timestamp(target_time)
    timestamp, path = min(
        indexed_files,
        key=lambda item: abs(
            (item[0] - target_time).total_seconds()
        ),
    )
    difference = abs(
        (timestamp - target_time).total_seconds()
    )

    if difference > MAX_TARGET_TIME_DIFFERENCE_SEC:
        raise ValueError(
            f"Nearest AIA file is {difference:.1f}s from "
            f"target {target_time}."
        )

    return path, timestamp, float(difference)

## 6. Dense-sequence and micro-window recovery exports

In [ ]:
def recovery_patch_size(group):
    recovery_class = str(group["recovery_class"].iloc[0])
    margin = (
        CROP_RECOVERY_MARGIN_ARCSEC
        if recovery_class == "crop_patch_boundary"
        else NORMAL_RECOVERY_MARGIN_ARCSEC
    )

    max_target_box = max(
        target_geometry(row)["box_arcsec"]
        for _, row in group.iterrows()
    )

    return float(
        np.clip(
            max_target_box + margin,
            MIN_RECOVERY_PATCH_ARCSEC,
            MAX_RECOVERY_PATCH_ARCSEC,
        )
    )


def export_patch_sequence(
    target_frame,
    wavelength,
    destination,
    start,
    duration_minutes,
    cadence_minutes,
    reference_row,
    patch_size,
    export_label,
):
    destination.mkdir(parents=True, exist_ok=True)

    existing = sorted(destination.glob("*.fits"))
    if existing and files_cover_targets(existing, target_frame):
        print(
            f"♻️ Reusing {len(existing)} recovery files for "
            f"{wavelength} Å ({export_label})"
        )
        return existing, {
            "request_id": "cached",
            "query": None,
            "patch_size_arcsec": patch_size,
            "export_label": export_label,
        }

    geometry = target_geometry(reference_row)

    query_string = (
        "aia.lev1_euv_12s"
        f"[{format_query_time(start)}/"
        f"{int(duration_minutes)}m@{int(cadence_minutes)}m]"
        f"[{int(wavelength)}]"
        "{image}"
    )

    process = {
        "im_patch": {
            "t_ref": format_query_time(reference_row["T_REC_dt"]),
            "t": 0,
            "r": 0,
            "c": 0,
            "locunits": "arcsec",
            "boxunits": "arcsec",
            "x": geometry["x_arcsec"],
            "y": geometry["y_arcsec"],
            "width": patch_size,
            "height": patch_size,
        }
    }

    print("Query:", query_string)
    print(
        "Patch:",
        patch_size,
        "arcsec | targets:",
        len(target_frame),
        "|",
        export_label,
    )

    request = submit_export_retry_safe(
        query_string,
        process,
    )
    request.download(destination, timeout=600)

    fits_files = sorted(destination.glob("*.fits"))
    if not fits_files:
        raise FileNotFoundError(
            f"No FITS files downloaded for {wavelength} Å "
            f"({export_label})."
        )

    metadata = {
        "request_id": request.id,
        "query": query_string,
        "wavelength": int(wavelength),
        "patch_size_arcsec": patch_size,
        "export_label": export_label,
        "n_files": len(fits_files),
        "email": JSOC_EMAIL,
    }
    with (destination / "export_metadata.json").open("w") as handle:
        json.dump(metadata, handle, indent=2)

    time.sleep(JSOC_COOLDOWN_SEC)
    return fits_files, metadata


def uncovered_rows(files, target_frame):
    if not files:
        return target_frame.copy()

    indexed = index_downloaded_files(files)
    deltas = target_deltas(indexed, target_frame)

    return target_frame[
        target_frame["sample_id"].astype(str).map(deltas)
        > MAX_TARGET_TIME_DIFFERENCE_SEC
    ].copy()


def obtain_recovery_files(
    group_id,
    original_block_id,
    recovery_class,
    group,
    wavelength,
):
    recovery_group_dir = (
        LOCAL_TEMP
        / re.sub(r"[^A-Za-z0-9_.-]+", "_", group_id)
        / str(wavelength)
    )
    recovery_group_dir.mkdir(parents=True, exist_ok=True)

    # Reuse retained production FITS only for time-coverage failures.
    # Crop-boundary failures require a larger server-side patch.
    source_files = []
    if recovery_class == "block_download_or_coverage":
        source_wave_dir = (
            SOURCE_TEMP_ROOT
            / str(original_block_id)
            / str(wavelength)
        )
        if source_wave_dir.exists():
            source_files = sorted(
                source_wave_dir.rglob("*.fits")
            )

    new_files = sorted(recovery_group_dir.rglob("*.fits"))
    combined = source_files + new_files

    if combined and files_cover_targets(combined, group):
        print(
            f"♻️ Existing files already cover {wavelength} Å: "
            f"{len(source_files)} source + {len(new_files)} recovery"
        )
        return combined, {
            "request_id": "cached",
            "strategy": "retained_production_cache",
        }

    group = group.sort_values("T_REC_dt").reset_index(drop=True)
    start = pd.Timestamp(group["T_REC_dt"].min())
    end = pd.Timestamp(group["T_REC_dt"].max())
    padded_start = start - pd.Timedelta(
        minutes=RECOVERY_SEQUENCE_CADENCE_MIN
    )
    padded_end = end + pd.Timedelta(
        minutes=RECOVERY_SEQUENCE_CADENCE_MIN
    )
    duration_minutes = max(
        RECOVERY_SEQUENCE_CADENCE_MIN,
        int(
            math.ceil(
                (padded_end - padded_start).total_seconds()
                / 60.0
            )
        ),
    )

    midpoint = start + (end - start) / 2
    reference_index = (
        group["T_REC_dt"] - midpoint
    ).abs().idxmin()
    reference_row = group.loc[reference_index]
    patch_size = recovery_patch_size(group)

    dense_dir = recovery_group_dir / "dense_12min"
    dense_files, dense_meta = export_patch_sequence(
        target_frame=group,
        wavelength=wavelength,
        destination=dense_dir,
        start=padded_start,
        duration_minutes=duration_minutes,
        cadence_minutes=RECOVERY_SEQUENCE_CADENCE_MIN,
        reference_row=reference_row,
        patch_size=patch_size,
        export_label="dense_12min_recovery",
    )

    combined = source_files + dense_files
    missing_after_dense = uncovered_rows(combined, group)

    request_ids = []
    if dense_meta.get("request_id"):
        request_ids.append(str(dense_meta["request_id"]))

    if len(missing_after_dense):
        print(
            f"{wavelength} Å still has "
            f"{len(missing_after_dense)} uncovered targets. "
            "Starting micro-window fallback."
        )

    for _, row in missing_after_dense.iterrows():
        target_time = pd.Timestamp(row["T_REC_dt"])
        sample_dir = (
            recovery_group_dir
            / "micro_windows"
            / str(row["sample_id"])
        )

        micro_start = target_time - pd.Timedelta(
            minutes=MICRO_WINDOW_BEFORE_MIN
        )

        micro_files, micro_meta = export_patch_sequence(
            target_frame=pd.DataFrame([row]),
            wavelength=wavelength,
            destination=sample_dir,
            start=micro_start,
            duration_minutes=MICRO_WINDOW_DURATION_MIN,
            cadence_minutes=MICRO_WINDOW_CADENCE_MIN,
            reference_row=row,
            patch_size=patch_size,
            export_label="target_micro_window",
        )
        combined.extend(micro_files)

        if micro_meta.get("request_id"):
            request_ids.append(str(micro_meta["request_id"]))

    combined = sorted(set(Path(item) for item in combined))

    if not files_cover_targets(combined, group):
        remaining = uncovered_rows(combined, group)
        raise RuntimeError(
            f"{wavelength} Å recovery still misses "
            f"{len(remaining)} target timestamps: "
            f"{remaining['sample_id'].head(10).tolist()}"
        )

    return combined, {
        "request_id": ",".join(
            sorted(set(request_ids))
        ) or "cached",
        "strategy": (
            "retained_cache+dense_12min+micro_fallback"
        ),
        "patch_size_arcsec": patch_size,
    }

## 7. Restore checkpoints and discover completed GCP objects

In [ ]:
def download_if_exists(gcp_path, local_path):
    if not gcp_exists(gcp_path):
        return False
    run_command(
        ["gcloud", "storage", "cp", gcp_path, str(local_path)],
        check=True,
    )
    return True


download_if_exists(
    f"{GCP_WORKER_META}/{LOCAL_RECOVERY_LOG.name}",
    LOCAL_RECOVERY_LOG,
)
download_if_exists(
    f"{GCP_WORKER_META}/{LOCAL_GROUP_LOG.name}",
    LOCAL_GROUP_LOG,
)

recovery_log = (
    pd.read_csv(LOCAL_RECOVERY_LOG, low_memory=False)
    if LOCAL_RECOVERY_LOG.exists()
    and LOCAL_RECOVERY_LOG.stat().st_size > 0
    else pd.DataFrame()
)
group_log = (
    pd.read_csv(LOCAL_GROUP_LOG, low_memory=False)
    if LOCAL_GROUP_LOG.exists()
    and LOCAL_GROUP_LOG.stat().st_size > 0
    else pd.DataFrame()
)

listing = run_command(
    [
        "gcloud",
        "storage",
        "ls",
        "--recursive",
        GCP_OUTPUT_ROOT,
    ],
    check=False,
)
completed_sample_ids = {
    Path(line.strip()).stem
    for line in listing.stdout.splitlines()
    if line.strip().endswith(".npz")
}

print("Completed 2026 GCP samples:", len(completed_sample_ids))
print("Recovery log rows:", len(recovery_log))
print("Recovery group log rows:", len(group_log))

## 8. Process recovery groups

In [ ]:
def append_checkpoint(
    frame,
    row,
    local_path,
    gcp_path,
    dedupe_column,
):
    updated = pd.concat(
        [frame, pd.DataFrame([row])],
        ignore_index=True,
    )
    updated = updated.drop_duplicates(
        subset=[dedupe_column],
        keep="last",
    )
    updated.to_csv(local_path, index=False)
    upload_verified(local_path, gcp_path)
    return updated


def process_recovery_group(group_row, group, recovery_log):
    started = time.time()

    group_id = str(group_row["recovery_group_id"])
    original_block_id = str(group_row["block_id"])
    recovery_class = str(group_row["recovery_class"])

    pending = group[
        ~group["sample_id"].astype(str).isin(completed_sample_ids)
    ].copy()

    if len(pending) == 0:
        return recovery_log, {
            "recovery_group_id": group_id,
            "original_block_id": original_block_id,
            "recovery_class": recovery_class,
            "status": "already_complete",
            "n_targets": len(group),
            "n_pending_at_start": 0,
            "n_saved_this_run": 0,
            "elapsed_minutes": 0.0,
            "message": "all selected samples already exist in GCP",
        }

    if free_disk_gb(LOCAL_ROOT) < MIN_FREE_DISK_GB:
        raise RuntimeError(
            f"Free disk below {MIN_FREE_DISK_GB} GB."
        )

    wavelength_indices = {}
    wavelength_metadata = {}

    for wavelength in AIA_WAVELENGTHS:
        print("\n" + "-" * 76)
        print(
            group_id,
            "| wavelength",
            wavelength,
            "| pending",
            len(pending),
        )

        files, export_meta = obtain_recovery_files(
            group_id=group_id,
            original_block_id=original_block_id,
            recovery_class=recovery_class,
            group=group,
            wavelength=wavelength,
        )
        wavelength_indices[wavelength] = (
            index_downloaded_files(files)
        )
        wavelength_metadata[wavelength] = export_meta

    saved_this_group = 0

    for _, row in pending.iterrows():
        sample_id = str(row["sample_id"])
        target_time = pd.Timestamp(row["T_REC_dt"])
        channels = []
        channel_metadata = {}

        try:
            for wavelength in AIA_WAVELENGTHS:
                path, used_time, delta_seconds = nearest_file(
                    wavelength_indices[wavelength],
                    target_time,
                )
                channel, crop_meta = crop_target_from_patch(
                    path,
                    row,
                )
                channels.append(channel)

                channel_metadata[str(wavelength)] = {
                    "source_file": path.name,
                    "used_time": str(used_time),
                    "delta_seconds": delta_seconds,
                    "request_id": (
                        wavelength_metadata[wavelength]
                        .get("request_id")
                    ),
                    "strategy": (
                        wavelength_metadata[wavelength]
                        .get("strategy")
                    ),
                    "crop": crop_meta,
                }

            tensor = np.stack(
                channels,
                axis=-1,
            ).astype(np.float32)

            if tensor.shape != (IMAGE_SIZE, IMAGE_SIZE, 6):
                raise ValueError(
                    f"Unexpected shape: {tensor.shape}"
                )
            if not np.isfinite(tensor).all():
                raise ValueError(
                    "Tensor contains NaN or infinity."
                )
            if tensor.min() < 0 or tensor.max() > 1:
                raise ValueError(
                    "Tensor values are outside [0, 1]."
                )

            local_npz = LOCAL_OUTPUT / f"{sample_id}.npz"

            np.savez_compressed(
                local_npz,
                x=tensor,
                y=np.array(
                    int(row["label_48h_final"]),
                    dtype=np.int64,
                ),
                sample_id=np.array(sample_id),
                T_REC_dt=np.array(str(target_time)),
                HARPNUM=np.array(
                    int(row["HARPNUM"]),
                    dtype=np.int64,
                ),
                NOAA_AR_clean=np.array(
                    int(row["NOAA_AR_clean"]),
                    dtype=np.int64,
                ),
                label_48h_final=np.array(
                    int(row["label_48h_final"]),
                    dtype=np.int64,
                ),
                wavelengths=np.array(
                    AIA_WAVELENGTHS,
                    dtype=np.int64,
                ),
                source=np.array(
                    "JSOC 2026 missing-only recovery"
                ),
                original_block_id=np.array(
                    original_block_id
                ),
                recovery_class=np.array(
                    recovery_class
                ),
                channel_metadata=np.array(
                    json.dumps(channel_metadata)
                ),
            )

            gcp_npz = f"{GCP_OUTPUT_ROOT}/{local_npz.name}"
            upload_verified(local_npz, gcp_npz)
            completed_sample_ids.add(sample_id)
            saved_this_group += 1

            recovery_row = {
                "sample_id": sample_id,
                "recovery_group_id": group_id,
                "original_block_id": original_block_id,
                "recovery_class": recovery_class,
                "T_REC_dt": str(target_time),
                "HARPNUM": int(row["HARPNUM"]),
                "NOAA_AR_clean": int(
                    row["NOAA_AR_clean"]
                ),
                "label_48h_final": int(
                    row["label_48h_final"]
                ),
                "status": "saved",
                "shape": str(tensor.shape),
                "gcp_path": gcp_npz,
                "message": "success",
                "updated_at_utc": (
                    pd.Timestamp.utcnow().isoformat()
                ),
            }

            recovery_log = append_checkpoint(
                recovery_log,
                recovery_row,
                LOCAL_RECOVERY_LOG,
                (
                    f"{GCP_WORKER_META}/"
                    f"{LOCAL_RECOVERY_LOG.name}"
                ),
                "sample_id",
            )

            local_npz.unlink(missing_ok=True)
            print("✅", sample_id)

        except Exception as error:
            recovery_row = {
                "sample_id": sample_id,
                "recovery_group_id": group_id,
                "original_block_id": original_block_id,
                "recovery_class": recovery_class,
                "T_REC_dt": str(target_time),
                "HARPNUM": int(row["HARPNUM"]),
                "NOAA_AR_clean": int(
                    row["NOAA_AR_clean"]
                ),
                "label_48h_final": int(
                    row["label_48h_final"]
                ),
                "status": "error",
                "shape": None,
                "gcp_path": None,
                "message": repr(error),
                "updated_at_utc": (
                    pd.Timestamp.utcnow().isoformat()
                ),
            }

            recovery_log = append_checkpoint(
                recovery_log,
                recovery_row,
                LOCAL_RECOVERY_LOG,
                (
                    f"{GCP_WORKER_META}/"
                    f"{LOCAL_RECOVERY_LOG.name}"
                ),
                "sample_id",
            )
            print("❌", sample_id, repr(error))

    elapsed_minutes = (time.time() - started) / 60.0

    current_errors = recovery_log[
        (
            recovery_log["recovery_group_id"]
            == group_id
        )
        & (
            recovery_log["status"]
            .astype(str)
            .str.lower()
            == "error"
        )
    ] if len(recovery_log) else pd.DataFrame()

    return recovery_log, {
        "recovery_group_id": group_id,
        "original_block_id": original_block_id,
        "recovery_class": recovery_class,
        "status": "completed",
        "n_targets": len(group),
        "n_pending_at_start": len(pending),
        "n_saved_this_run": saved_this_group,
        "elapsed_minutes": round(
            elapsed_minutes,
            3,
        ),
        "message": (
            "success"
            if len(current_errors) == 0
            else (
                f"{len(current_errors)} sample errors "
                "retained for review"
            )
        ),
    }


for position, group_row in group_table.iterrows():
    group_id = str(group_row["recovery_group_id"])
    group = selected_recovery_df[
        selected_recovery_df["recovery_group_id"]
        == group_id
    ].sort_values("T_REC_dt").copy()

    print("\n" + "=" * 96)
    print(
        f"RECOVERY GROUP {position + 1}/{len(group_table)} | "
        f"{group_id} | targets={len(group)}"
    )
    print("=" * 96)

    try:
        recovery_log, result = process_recovery_group(
            group_row,
            group,
            recovery_log,
        )
    except Exception as error:
        result = {
            "recovery_group_id": group_id,
            "original_block_id": str(
                group_row["block_id"]
            ),
            "recovery_class": str(
                group_row["recovery_class"]
            ),
            "status": "error",
            "n_targets": len(group),
            "n_pending_at_start": None,
            "n_saved_this_run": 0,
            "elapsed_minutes": None,
            "message": repr(error),
        }
        print("GROUP ERROR:", repr(error))

    group_log = append_checkpoint(
        group_log,
        result,
        LOCAL_GROUP_LOG,
        f"{GCP_WORKER_META}/{LOCAL_GROUP_LOG.name}",
        "recovery_group_id",
    )
    display(pd.DataFrame([result]))

    gc.collect()

print("\nRecovery run finished.")

## 9. Final audit

In [ ]:
fresh_listing = run_command(
    [
        "gcloud",
        "storage",
        "ls",
        "--recursive",
        GCP_OUTPUT_ROOT,
    ],
    check=False,
)
actual_ids = {
    Path(line.strip()).stem
    for line in fresh_listing.stdout.splitlines()
    if line.strip().endswith(".npz")
}

all_recovery_ids = set(
    recovery_df["sample_id"].astype(str)
)
selected_ids = set(
    selected_recovery_df["sample_id"].astype(str)
)
recovered_all = all_recovery_ids.intersection(actual_ids)
recovered_selected = selected_ids.intersection(actual_ids)

remaining_all = sorted(all_recovery_ids - actual_ids)
remaining_selected = sorted(selected_ids - actual_ids)

audit = pd.DataFrame(
    {
        "metric": [
            "selected_recovery_scope",
            "selected_completed_in_gcp",
            "selected_remaining",
            "all_2026_recovery_scope",
            "all_recovery_completed_in_gcp",
            "all_recovery_remaining",
            "overall_2026_objects_in_gcp",
            "overall_2026_expected",
        ],
        "value": [
            len(selected_ids),
            len(recovered_selected),
            len(remaining_selected),
            len(all_recovery_ids),
            len(recovered_all),
            len(remaining_all),
            len(actual_ids),
            3201,
        ],
    }
)

display(audit)

remaining_path = (
    LOCAL_META
    / f"remaining_recovery_ids_{WORKER_ID}.txt"
)
remaining_path.write_text(
    "\n".join(remaining_all),
    encoding="utf-8",
)
upload_verified(
    remaining_path,
    f"{GCP_WORKER_META}/{remaining_path.name}",
)

audit_path = LOCAL_META / f"recovery_audit_{WORKER_ID}.csv"
audit.to_csv(audit_path, index=False)
upload_verified(
    audit_path,
    f"{GCP_WORKER_META}/{audit_path.name}",
)

print("Selected remaining:", len(remaining_selected))
print("All recovery remaining:", len(remaining_all))
print("Overall 2026 GCP objects:", len(actual_ids))
print("Unexpected objects:", len(actual_ids - set(metadata_2026["sample_id"])))

## 10. Acceptance gate

Before switching from `CANARY` to `PRODUCTION`, confirm:

1. At least one sample from each recovery class was saved.
2. Every recovered tensor has shape `(512, 512, 6)`.
3. All values are finite and within `[0, 1]`.
4. Every selected AIA timestamp is within 180 seconds of its SHARP target.
5. The enlarged-patch HARP 14305 crops no longer leave the server-side patch.
6. No unexpected 2026 sample IDs were created.
7. The 2025 shard workers were not interrupted.

Production launch variables:

```bash
export TARGET_YEAR=2026
export JSOC_EMAIL="worky4work@gmail.com"
export WORKER_ID="aia2026-recovery"
export RECOVERY_MODE="PRODUCTION"
```